#7. Modelos de Regressão


Este notebook explora diferentes modelos de regressão para análise de dados, incluindo Regressão Linear e Lasso, utilizando técnicas de pré-processamento como transformação Box-Cox e seleção de variáveis. O objetivo é avaliar o desempenho dos modelos em termos de métricas como RMSE e R², tanto nos datasets completos quanto após a remoção de outliers das questões do Enem (vetorizadas).

In [ ]:
# Importando Dependências para Modelos de Regressão
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats
import statsmodels.api as sm

from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error
from sklearn.linear_model import Lasso, LassoCV
#import scipy

In [ ]:
# Leitura dos dados
enem_data = pd.read_pickle("../data/final/enem_data_embeddings.pkl")
enem_data.head()

---

## 6.1. Regressão Linear - 300 dimensões

### 6.1.1. Dataset Completo


In [ ]:
# Coletando os dados
X = [np.array(embedding) for embedding in enem_data["enunciado_embbedings_word2vec"]]
y = enem_data["dificuldade"]

In [ ]:
# Aplicando Transformações
add_list = [(y.min()*(-1))+1]*len(y)
y = y + add_list

# Aplicando Boxcox
y, best_lambda = stats.boxcox(y)
print(best_lambda)

0.6375985567450835


In [ ]:
# Separando em conjunto de treino e teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

In [ ]:
# Adicionando constante
X_train = sm.add_constant(X_train)
X_test = sm.add_constant(X_test)

In [ ]:
# Criando o modelo e realizando predição
model = sm.OLS(y_train, X_train).fit()
pred = model.predict(X_test)

In [ ]:
# Cálculo do RMSE
rms = root_mean_squared_error(y_test, pred)
print('RMSE', rms)

RMSE 1.0504498350168308


In [ ]:
# Visualização do modelo e resultados
print(model.summary(alpha=0.05))

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.710
Model:                            OLS   Adj. R-squared:                  0.110
Method:                 Least Squares   F-statistic:                     1.184
Date:                Thu, 22 May 2025   Prob (F-statistic):              0.125
Time:                        19:27:35   Log-Likelihood:                -80.002
No. Observations:                 446   AIC:                             762.0
Df Residuals:                     145   BIC:                             1996.
Df Model:                         300                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          2.8570      0.941      3.037      0.0

### 6.1.3. Modelo com Corte (-3,3)


In [ ]:
enem_filtered = enem_data.copy()
enem_filtered = enem_filtered[
    (enem_filtered["dificuldade"] >= -3) & (enem_filtered["dificuldade"] <= 3)
]
enem_filtered["dificuldade"].describe()

In [ ]:
# Coletando os dados
X = [np.array(embedding) for embedding in enem_filtered["enunciado_embbedings_word2vec"]]
y = enem_filtered["dificuldade"]

In [ ]:
# Aplicando Transformações
add_list = [(y.min()*(-1))+1]*len(y)
y = y + add_list

# Aplicando Boxcox
y, best_lambda = stats.boxcox(y)
print(best_lambda)

1.7873995037544776


In [ ]:
# Dividindo os dados em treino e teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

In [ ]:
# Adicionando constante
X_train = sm.add_constant(X_train)
X_test = sm.add_constant(X_test)

In [ ]:
# Criando o modelo e realizando a predição
model = sm.OLS(y_train, X_train).fit()
pred = model.predict(X_test)

In [ ]:
# Cálculo do RMSE
rms = root_mean_squared_error(y_test, pred)
print("RMSE:", rms)

RMSE 4.048974956373059


In [ ]:
print(model.summary(alpha = 0.05))

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.759
Model:                            OLS   Adj. R-squared:                  0.217
Method:                 Least Squares   F-statistic:                     1.399
Date:                Thu, 22 May 2025   Prob (F-statistic):             0.0137
Time:                        19:27:35   Log-Likelihood:                -652.64
No. Observations:                 434   AIC:                             1907.
Df Residuals:                     133   BIC:                             3133.
Df Model:                         300                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          4.3232      4.078      1.060      0.2

## 6.2. Regressão Lasso - 300 dimensões


### 6.2.1. Dataset Completo

In [ ]:
# Coletando os dados
X = [np.array(embedding) for embedding in enem_data["enunciado_embbedings_word2vec"]]
y = enem_data["dificuldade"]

In [ ]:
# Aplicando Transformações
add_list = [(y.min()*(-1))+1]*len(y)
y = y + add_list

# Aplicando Boxcox
y, best_lambda = stats.boxcox(y)
print(best_lambda)

1.7873995037544776


In [ ]:
# Separando em conjunto de treino e teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

In [ ]:
model_lasso = LassoCV(alphas=[0.0001, 0.001, 0.01, 0.1, 1, 10], random_state=0).fit(
    X_train, y_train
)
pred_lasso = model_lasso.predict(X_test)

print("melhor r2 score:", model_lasso.score(X_train, y_train))
print("melhor alpha:", model_lasso.alpha_)
print("RMSE com o alpha escolhido:", root_mean_squared_error(y_test, pred_lasso))

melhor r2 score: 0.14278869322754884
melhor alpha: 0.01
RMSE com o alpha escolhido: 2.26804106529649


### 6.2.2. Modelo com Corte (-3, 3)

In [ ]:
enem_filtered = enem_data.copy()
enem_filtered = enem_filtered[
    (enem_filtered["dificuldade"] >= -3) & (enem_filtered["dificuldade"] <= 3)
]
enem_filtered["dificuldade"].describe()

In [ ]:
# Coletando os dados
X = [np.array(embedding) for embedding in enem_filtered["enunciado_embbedings_word2vec"]]
y = enem_filtered["dificuldade"]

In [ ]:
# Aplicando Transformações
add_list = [(y.min()*(-1))+1]*len(y)
y = y + add_list

# Aplicando Boxcox
y, best_lambda = stats.boxcox(y)
print(best_lambda)

1.7873995037544776


In [ ]:
# Dividindo os dados em treino e teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

In [ ]:
model_lasso = LassoCV(alphas=[0.0001, 0.001, 0.01, 0.1, 1, 10, 100], random_state=0, tol=0.1).fit(
    X_train, y_train
)
pred_lasso = model_lasso.predict(X_test)

print("melhor r2 score:", model_lasso.score(X_train, y_train))
print("melhor alpha:", model_lasso.alpha_)
print("RMSE com o alpha escolhido:", root_mean_squared_error(y_test, pred_lasso))

melhor r2 score: 0.134062064100828
melhor alpha: 0.01
RMSE com o alpha escolhido: 2.20860448083319


## 6.3. Inclusão das provas Ledor - 300 dimensões

### 6.3.1. Regressão linear

In [ ]:
enem_data_ledor = pd.read_pickle("/content/drive/MyDrive/MED-enem/cleaned_data_ledor_vectors.pkl").dropna()
enem_data_ledor.head()

,numero_questao,enunciado,alternativas,nu_param_B,gabarito,year,enunciado_limpo,alternativas_limpo,vetor_enunciado,vetor_media
0,1,A atmosfera terrestre é composta pelos gases n...,A: reduzir o calor irradiado pela Terra median...,-1.70677,C,2009,absorver acontece alternativa anos aquecendo a...,A: calor industrialização irradiado mediante p...,"[[-0.072753, 0.136878, 0.160618, -0.442716, -0...","[0.016323044885853854, -0.032005971771842966, ..."
1,2,Analise a figura. Supondo que seja necessário ...,A: Concentração média de álcool no sangue ao l...,0.62043,D,2009,alternativa analise dar figura melhor necessár...,A: concentração dia longo média sangue álcool;...,"[[0.184257, -0.035359, 0.075923, 0.163333, -0....","[-0.0381658197465268, -0.027103909206661312, 0..."
2,3,Estima-se que haja atualmente no mundo 40 milh...,"A: induzir a imunidade, para proteger o organi...",2.07704,A,2009,acompanhamento aids antiviral atualmente caren...,A: contaminação imunidade induzir organismo pr...,"[[-0.117196, 0.170721, 0.426886, -0.014566, 0....","[0.0118313719691752, 0.012745790416374803, 0.0..."
3,4,"Em um experimento, preparou-se um conjunto de ...",A: os genótipos e os fenótipos idênticos.; B: ...,0.11500,B,2009,alguns amareladas apresentaram apresentava apó...,A: fenótipos genótipos idênticos; B: diferente...,"[[-0.000374, -0.127046, -0.401209, -0.020222, ...","[-0.015016743819265125, -0.01632628217041015, ..."
4,5,"Na linha de uma tradição antiga, o astrônomo g...",A: Ptolomeu apresentou as ideias mais valiosas...,0.21694,E,2009,afirmar afirmou alemão anos antiga astronômico...,A: antigas apresentou ideias ptolomeu serem tr...,"[[-0.074335, -0.090428, -0.019981, -0.153818, ...","[-0.02099789898974173, 0.008896578985698305, -..."


In [ ]:
# Coletando os dados
X = [np.array(embedding) for embedding in enem_data_ledor["enunciado_embbedings_word2vec"]]
y = enem_data_ledor["dificuldade"]

In [ ]:
# Aplicando Transformações
add_list = [(y.min() * (-1)) + 1] * len(y)
y = y + add_list

# Aplicando Boxcox
y, best_lambda = stats.boxcox(y)
print(best_lambda)

0.629376905725391


In [ ]:
# Separando em conjunto de treino e teste

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

In [ ]:
# Adicionando constante
X_train = sm.add_constant(X_train)
X_test = sm.add_constant(X_test)

In [ ]:
# Criando o modelo e realizando predição
model = sm.OLS(y_train, X_train).fit()
pred = model.predict(X_test)

RMSE 1.0011189094119426


In [ ]:
# Cálculo do RMSE
rms = root_mean_squared_error(y_test, pred)
print('RMSE', rms)

In [ ]:
print(model.summary(alpha = 0.05))

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.699
Model:                            OLS   Adj. R-squared:                  0.097
Method:                 Least Squares   F-statistic:                     1.161
Date:                Thu, 22 May 2025   Prob (F-statistic):              0.151
Time:                        19:27:39   Log-Likelihood:                -81.498
No. Observations:                 451   AIC:                             765.0
Df Residuals:                     150   BIC:                             2003.
Df Model:                         300                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          1.6162      1.022      1.581      0.1

### 6.3.2. Lasso

In [ ]:
# Coletando os dados
X = [np.array(embedding) for embedding in enem_data_ledor["enunciado_embbedings_word2vec"]]
y = enem_data_ledor["dificuldade"]

In [ ]:
# Aplicando Transformações
add_list = [(y.min() * (-1)) + 1] * len(y)
y = y + add_list

# Aplicando Boxcox
y, best_lambda = stats.boxcox(y)
print(best_lambda)

0.629376905725391


In [ ]:
# Separando em conjunto de treino e teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

In [ ]:
model_lasso = LassoCV(alphas=[0.0001, 0.001, 0.01, 0.1, 1, 10], random_state=0).fit(
    X_train, y_train
)
pred_lasso = model_lasso.predict(X_test)
print('melhor r2 score:', model_lasso.score(X_train, y_train))
print('melhor alpha:', model_lasso.alpha_)
print('RMSE com o alpha escolhido:', root_mean_squared_error(y_test, pred_lasso))

melhor r2 score: 0.20644803088589936
melhor alpha: 0.001
RMSE com o alpha escolhido: 0.5093419924430485


##6.4. Regressão linear - 100 dimensões

In [ ]:
enem_data_100 = pd.read_pickle("/content/drive/MyDrive/MED-enem/cleaned_data_vectors_100.pkl").dropna()
enem_data_100.head()

,numero_questao,enunciado,alternativas,nu_param_B,gabarito,ano,enunciado_limpo,alternativas_limpo,vetor_enunciado,vetor_media
0,91,Para realizar o desentupimento de tubulações d...,A: Al; B: Co; C: Cu(OH)2; D: Fe(OH)2; E: Pb,1.13773,A,2019,adicionada aumentando aumentar básico calor co...,A: al; B: co; C: cu oh; D: fe oh; E: pb,"[[0.163096, 0.127435, -0.249899, -0.089416, 0....","[0.08290613839012939, -0.07632909273338873, 0...."
1,92,As redes de alta tensão para transmissão de en...,A: Fazer o aterramento dos arames da cerca.; B...,1.45214,A,2019,alta animais aproximarem arame campo cerca cer...,A: arames aterramento cerca fazer; B: acrescen...,"[[0.263261, 0.06409, -0.264677, 0.473302, -0.1...","[-0.01782597419925225, -0.06925118028019102, 0..."
2,93,A esquistossomose (barriga-dʼágua) caracteriza...,A: impedir a penetração do parasita pela pele....,0.58402,E,2019,anticorpos após barriga baseia baço biomphalar...,A: impedir parasita pele penetração; B: caramu...,"[[0.169307, 0.133243, 0.04853, -0.009939, -0.1...","[0.10779110447650678, -0.06825856033206117, 0...."
3,94,"Em 1962, um jingle (vinheta musical) criado po...",A: Aquecer a casa e os corpos.; B: Evitar a en...,0.41535,C,2019,abriria animado apesar aquecer associar batia ...,A: aquecer casa corpos; B: casa corpos entrada...,"[[-0.343659, -0.018934, -0.158586, 0.218599, 0...","[0.02177992160146709, 0.014867862881681718, -0..."
4,95,Glicólise é um processo que ocorre nas células...,A: libera 112 kJ por mol de glicose.; B: liber...,2.21432,A,2019,alguns anaeróbico casos combustão completament...,A: glicose kj libera mol; B: glicose kj libera...,"[[-0.304121, -0.076877, -0.194382, 0.040576, -...","[0.02815826823253457, -0.0134348290711187, 0.0..."


###6.4.1 Dataset completo

In [ ]:
# Coletando os dados
X = [np.array(embedding) for embedding in enem_data_100["enunciado_embbedings_word2vec"]]
y = enem_data_100["dificuldade"]

In [ ]:
# Aplicando Transformações
add_list = [(y.min() * (-1)) + 1] * len(y)
y = y + add_list

# Aplicando Boxcox
y, best_lambda = stats.boxcox(y)
print(best_lambda)

0.6375985567450835


In [ ]:
# Separando em conjunto de treino e teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

In [ ]:
# Adicionando constante
X_train = sm.add_constant(X_train)
X_test = sm.add_constant(X_test)

In [ ]:
# Criando o modelo e realizando predição
model = sm.OLS(y_train, X_train).fit()
pred = model.predict(X_test)

RMSE 0.584520228126551


In [ ]:
# Cálculo do RMSE
rms = root_mean_squared_error(y_test, pred)
print('RMSE', rms)

In [ ]:
print(model.summary(alpha = 0.05))

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.270
Model:                            OLS   Adj. R-squared:                  0.058
Method:                 Least Squares   F-statistic:                     1.274
Date:                Thu, 22 May 2025   Prob (F-statistic):             0.0587
Time:                        19:27:40   Log-Likelihood:                -286.06
No. Observations:                 446   AIC:                             774.1
Df Residuals:                     345   BIC:                             1188.
Df Model:                         100                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          2.3947      0.511      4.689      0.0

###6.4.2. Modelo com Corte (-3,3)

In [ ]:
enem_filtered_100 = enem_data_100.copy()
enem_filtered_100 = enem_filtered_100[
    (enem_filtered_100["dificuldade"] >= -3) & (enem_filtered_100["dificuldade"] <= 3)
]
enem_filtered_100["dificuldade"].describe()

In [ ]:
# Coletando os dados
X = [np.array(embedding) for embedding in enem_filtered_100["enunciado_embbedings_word2vec"]]
y = enem_filtered_100["dificuldade"]

In [ ]:
# Aplicando Transformações
add_list = [(y.min() * (-1)) + 1] * len(y)
y = y + add_list

# Aplicando Boxcox
y, best_lambda = stats.boxcox(y)
print(best_lambda)

1.7873995037544776


In [ ]:
# Dividindo os dados em treino e teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

In [ ]:
# Adicionando constante
X_train = sm.add_constant(X_train)
X_test = sm.add_constant(X_test)

In [ ]:
# Criando o modelo e realizando predição
model = sm.OLS(y_train, X_train).fit()
pred = model.predict(X_test)

In [ ]:
# Cálculo do RMSE
rms = root_mean_squared_error(y_test, pred)
print("RMSE:", rms)

RMSE 2.4710302449839294


In [ ]:
print(model.summary(alpha = 0.05))

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.343
Model:                            OLS   Adj. R-squared:                  0.145
Method:                 Least Squares   F-statistic:                     1.735
Date:                Thu, 22 May 2025   Prob (F-statistic):           0.000158
Time:                        19:27:41   Log-Likelihood:                -870.76
No. Observations:                 434   AIC:                             1944.
Df Residuals:                     333   BIC:                             2355.
Df Model:                         100                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          7.3130      2.166      3.376      0.0

## 6.5. Regressão linear - 50 dimensões

In [ ]:
enem_data_50 = pd.read_pickle("/content/drive/MyDrive/MED-enem/cleaned_data_vectors_50.pkl").dropna()
enem_data_50.head()

,numero_questao,enunciado,alternativas,nu_param_B,gabarito,ano,enunciado_limpo,alternativas_limpo,vetor_enunciado,vetor_media
0,91,Para realizar o desentupimento de tubulações d...,A: Al; B: Co; C: Cu(OH)2; D: Fe(OH)2; E: Pb,1.13773,A,2019,adicionada aumentando aumentar básico calor co...,A: al; B: co; C: cu oh; D: fe oh; E: pb,"[[0.167646, 0.032219, 0.126166, -0.468676, -0....","[0.15969267525428602, 0.1656768145310411, 0.05..."
1,92,As redes de alta tensão para transmissão de en...,A: Fazer o aterramento dos arames da cerca.; B...,1.45214,A,2019,alta animais aproximarem arame campo cerca cer...,A: arames aterramento cerca fazer; B: acrescen...,"[[0.254412, 0.464122, -0.360113, -0.075608, -0...","[0.07740464107169268, 0.07946692328326978, 0.0..."
2,93,A esquistossomose (barriga-dʼágua) caracteriza...,A: impedir a penetração do parasita pela pele....,0.58402,E,2019,anticorpos após barriga baseia baço biomphalar...,A: impedir parasita pele penetração; B: caramu...,"[[-0.21228, 0.511012, -0.102683, -0.056822, 0....","[0.025618789104842825, 0.08897740299621466, 0...."
3,94,"Em 1962, um jingle (vinheta musical) criado po...",A: Aquecer a casa e os corpos.; B: Evitar a en...,0.41535,C,2019,abriria animado apesar aquecer associar batia ...,A: aquecer casa corpos; B: casa corpos entrada...,"[[-0.219446, -0.167518, -0.241492, 0.002842, -...","[-0.016323587822946992, -0.038351687000078315,..."
4,95,Glicólise é um processo que ocorre nas células...,A: libera 112 kJ por mol de glicose.; B: liber...,2.21432,A,2019,alguns anaeróbico casos combustão completament...,A: glicose kj libera mol; B: glicose kj libera...,"[[0.108379, -0.268218, -0.0819, -0.138854, -0....","[0.01706921740821222, 0.138990388950333, 0.026..."


###6.5.1. Dataset completo

In [ ]:
# Coletando os dados
X = [np.array(embedding) for embedding in enem_data_50["enunciado_embbedings_word2vec"]]
y = enem_data_50["dificuldade"]

In [ ]:
# Aplicando Transformações
add_list = [(y.min() * (-1)) + 1] * len(y)
y = y + add_list

# Aplicando Boxcox
y, best_lambda = stats.boxcox(y)
print(best_lambda)

0.6375985567450835


In [ ]:
# Separando em conjunto de treino e teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

In [ ]:
# Adicionando constante
X_train = sm.add_constant(X_train)
X_test = sm.add_constant(X_test)

In [ ]:
# Criando o modelo e realizando predição
model = sm.OLS(y_train, X_train).fit()
pred = model.predict(X_test)

In [ ]:
# Cálculo do RMSE
rms = root_mean_squared_error(y_test, pred)
print('RMSE', rms)

RMSE 0.54070010642622


In [ ]:
print(model.summary(alpha = 0.05))

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.173
Model:                            OLS   Adj. R-squared:                  0.068
Method:                 Least Squares   F-statistic:                     1.648
Date:                Thu, 22 May 2025   Prob (F-statistic):            0.00524
Time:                        19:27:41   Log-Likelihood:                -313.88
No. Observations:                 446   AIC:                             729.8
Df Residuals:                     395   BIC:                             938.9
Df Model:                          50                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          1.9119      0.392      4.871      0.0

### 6.5.2. Modelo com Corte (-3,3)

In [ ]:
enem_filtered_50 = enem_data_50.copy()
enem_filtered_50 = enem_filtered_50[
    (enem_filtered_50["dificuldade"] >= -3) & (enem_filtered_50["dificuldade"] <= 3)
]
enem_filtered_50["dificuldade"].describe()

In [ ]:
# Coletando os dados
X = [np.array(embedding) for embedding in enem_filtered_50["enunciado_embbedings_word2vec"]]
y = enem_filtered_50["dificuldade"]

In [ ]:
# Aplicando Transformações
add_list = [(y.min() * (-1)) + 1] * len(y)
y = y + add_list

# Aplicando Boxcox
y, best_lambda = stats.boxcox(y)
print(best_lambda)

In [ ]:
# Dividindo os dados em treino e teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

In [ ]:
# Adicionando constante
X_train = sm.add_constant(X_train)
X_test = sm.add_constant(X_test)

In [ ]:
# Criando o modelo e realizando predição
model = sm.OLS(y_train, X_train).fit()
pred = model.predict(X_test)

In [ ]:
# Cálculo do RMSE
rms = root_mean_squared_error(y_test, pred)
print("RMSE:", rms)

In [ ]:
print(model.summary(alpha = 0.05))

## 6.6. Regressão Lasso - 100 dimensões


In [ ]:
enem_data_100 = pd.read_pickle("/content/drive/MyDrive/MED-enem/cleaned_data_vectors_100.pkl").dropna()
enem_data_100.head()

,numero_questao,enunciado,alternativas,nu_param_B,gabarito,ano,enunciado_limpo,alternativas_limpo,vetor_enunciado,vetor_media
0,91,Para realizar o desentupimento de tubulações d...,A: Al; B: Co; C: Cu(OH)2; D: Fe(OH)2; E: Pb,1.13773,A,2019,adicionada aumentando aumentar básico calor co...,A: al; B: co; C: cu oh; D: fe oh; E: pb,"[[0.163096, 0.127435, -0.249899, -0.089416, 0....","[0.08290613839012939, -0.07632909273338873, 0...."
1,92,As redes de alta tensão para transmissão de en...,A: Fazer o aterramento dos arames da cerca.; B...,1.45214,A,2019,alta animais aproximarem arame campo cerca cer...,A: arames aterramento cerca fazer; B: acrescen...,"[[0.263261, 0.06409, -0.264677, 0.473302, -0.1...","[-0.01782597419925225, -0.06925118028019102, 0..."
2,93,A esquistossomose (barriga-dʼágua) caracteriza...,A: impedir a penetração do parasita pela pele....,0.58402,E,2019,anticorpos após barriga baseia baço biomphalar...,A: impedir parasita pele penetração; B: caramu...,"[[0.169307, 0.133243, 0.04853, -0.009939, -0.1...","[0.10779110447650678, -0.06825856033206117, 0...."
3,94,"Em 1962, um jingle (vinheta musical) criado po...",A: Aquecer a casa e os corpos.; B: Evitar a en...,0.41535,C,2019,abriria animado apesar aquecer associar batia ...,A: aquecer casa corpos; B: casa corpos entrada...,"[[-0.343659, -0.018934, -0.158586, 0.218599, 0...","[0.02177992160146709, 0.014867862881681718, -0..."
4,95,Glicólise é um processo que ocorre nas células...,A: libera 112 kJ por mol de glicose.; B: liber...,2.21432,A,2019,alguns anaeróbico casos combustão completament...,A: glicose kj libera mol; B: glicose kj libera...,"[[-0.304121, -0.076877, -0.194382, 0.040576, -...","[0.02815826823253457, -0.0134348290711187, 0.0..."


### 6.6.1. Dataset completo

In [ ]:
# Coletando os dados
X = [np.array(embedding) for embedding in enem_data_100["enunciado_embbedings_word2vec"]]
y = enem_data_100["dificuldade"]

In [ ]:
# Aplicando Transformações
add_list = [(y.min() * (-1)) + 1] * len(y)
y = y + add_list

# Aplicando Boxcox
y, best_lambda = stats.boxcox(y)
print(best_lambda)

0.6375985567450835


In [ ]:
# Separando em conjunto de treino e teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

In [ ]:
model_lasso = LassoCV(alphas=[0.0001, 0.001, 0.01, 0.1, 1, 10], random_state=0).fit(
    X_train, y_train
)
pred_lasso = model_lasso.predict(X_test)

print('melhor r2 score:', model_lasso.score(X_train, y_train))
print('melhor alpha:', model_lasso.alpha_)
print('RMSE com o alpha escolhido:', root_mean_squared_error(y_test, pred_lasso))

melhor r2 score: 0.1742266934562141
melhor alpha: 0.001
RMSE com o alpha escolhido: 0.510679154079445


###6.6.2. Modelo com Corte (-3,3)

In [ ]:
enem_filtered_100 = enem_filtered_100.copy()
enem_filtered_100 = enem_filtered_100[
    (enem_filtered_100["dificuldade"] >= -3) & (enem_filtered_100["dificuldade"] <= 3)
]
enem_filtered_100["dificuldade"].describe()

In [ ]:
# Coletando os dados
X = [np.array(embedding) for embedding in enem_filtered_100["enunciado_embbedings_word2vec"]]
y = enem_filtered_100["dificuldade"]

In [ ]:
# Aplicando Transformações
add_list = [(y.min() * (-1)) + 1] * len(y)
y = y + add_list

# Aplicando Boxcox
y, best_lambda = stats.boxcox(y)
print(best_lambda)

1.7873995037544776


In [ ]:
# Dividindo os dados em treino e teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

In [ ]:
model_lasso = LassoCV(alphas=[0.0001, 0.001, 0.01, 0.1, 1, 10], random_state=0).fit(
    X_train, y_train
)
pred_lasso = model_lasso.predict(X_test)

print('melhor r2 score:', model_lasso.score(X_train, y_train))
print('melhor alpha:', model_lasso.alpha_)
print('RMSE com o alpha escolhido:', root_mean_squared_error(y_test, pred_lasso))

melhor r2 score: 0.12993724679725704
melhor alpha: 0.01
RMSE com o alpha escolhido: 2.2465387816328306


## 6.7. Regresão Lasso - 50 dimensões


In [ ]:
enem_data_50 = pd.read_pickle("/content/drive/MyDrive/MED-enem/cleaned_data_vectors_100.pkl").dropna()
enem_data_50.head()

,numero_questao,enunciado,alternativas,nu_param_B,gabarito,ano,enunciado_limpo,alternativas_limpo,vetor_enunciado,vetor_media
0,91,Para realizar o desentupimento de tubulações d...,A: Al; B: Co; C: Cu(OH)2; D: Fe(OH)2; E: Pb,1.13773,A,2019,adicionada aumentando aumentar básico calor co...,A: al; B: co; C: cu oh; D: fe oh; E: pb,"[[0.167646, 0.032219, 0.126166, -0.468676, -0....","[0.15969267525428602, 0.1656768145310411, 0.05..."
1,92,As redes de alta tensão para transmissão de en...,A: Fazer o aterramento dos arames da cerca.; B...,1.45214,A,2019,alta animais aproximarem arame campo cerca cer...,A: arames aterramento cerca fazer; B: acrescen...,"[[0.254412, 0.464122, -0.360113, -0.075608, -0...","[0.07740464107169268, 0.07946692328326978, 0.0..."
2,93,A esquistossomose (barriga-dʼágua) caracteriza...,A: impedir a penetração do parasita pela pele....,0.58402,E,2019,anticorpos após barriga baseia baço biomphalar...,A: impedir parasita pele penetração; B: caramu...,"[[-0.21228, 0.511012, -0.102683, -0.056822, 0....","[0.025618789104842825, 0.08897740299621466, 0...."
3,94,"Em 1962, um jingle (vinheta musical) criado po...",A: Aquecer a casa e os corpos.; B: Evitar a en...,0.41535,C,2019,abriria animado apesar aquecer associar batia ...,A: aquecer casa corpos; B: casa corpos entrada...,"[[-0.219446, -0.167518, -0.241492, 0.002842, -...","[-0.016323587822946992, -0.038351687000078315,..."
4,95,Glicólise é um processo que ocorre nas células...,A: libera 112 kJ por mol de glicose.; B: liber...,2.21432,A,2019,alguns anaeróbico casos combustão completament...,A: glicose kj libera mol; B: glicose kj libera...,"[[0.108379, -0.268218, -0.0819, -0.138854, -0....","[0.01706921740821222, 0.138990388950333, 0.026..."


### 6.7.1. Dataset completo

In [ ]:
# Coletando os dados
X = [np.array(embedding) for embedding in enem_data_50["enunciado_embbedings_word2vec"]]
y = enem_data_50["dificuldade"]

In [ ]:
# Aplicando Transformações
add_list = [(y.min() * (-1)) + 1] * len(y)
y = y + add_list

# Aplicando Boxcox
y, best_lambda = stats.boxcox(y)
print(best_lambda)

0.6375985567450835


In [ ]:
# Separando em conjunto de treino e teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

In [ ]:
model_lasso = LassoCV(alphas=[0.0001, 0.001, 0.01, 0.1, 1, 10], random_state=0).fit(
    X_train, y_train
)
pred_lasso = model_lasso.predict(X_test)

print('melhor r2 score:', model_lasso.score(X_train, y_train))
print('melhor alpha:', model_lasso.alpha_)
print('RMSE com o alpha escolhido:', root_mean_squared_error(y_test, pred_lasso))

melhor r2 score: 0.13436211723982172
melhor alpha: 0.001
RMSE com o alpha escolhido: 0.5095468517592282


###6.7.2. Modelo com Corte (-3,3)

In [ ]:
enem_filtered_50 = enem_filtered_50.copy()
enem_filtered_50 = enem_filtered_50[
    (enem_filtered_50["dificuldade"] >= -3) & (enem_filtered_50["dificuldade"] <= 3)
]
enem_filtered_50["dificuldade"].describe()

In [ ]:
# Coletando os dados
X = [np.array(embedding) for embedding in enem_filtered_50["enunciado_embbedings_word2vec"]]
y = enem_filtered_50["dificuldade"]

In [ ]:
# Aplicando Transformações
add_list = [(y.min() * (-1)) + 1] * len(y)
y = y + add_list

# Aplicando Boxcox
y, best_lambda = stats.boxcox(y)
print(best_lambda)

1.7873995037544776


In [ ]:
# Separando em conjunto de treino e teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

In [ ]:
model_lasso = LassoCV(alphas=[0.0001, 0.001, 0.01, 0.1, 1, 10], random_state=0).fit(
    X_train, y_train
)
pred_lasso = model_lasso.predict(X_test)

print('melhor r2 score:', model_lasso.score(X_train, y_train))
print('melhor alpha:', model_lasso.alpha_)
print('RMSE com o alpha escolhido:', root_mean_squared_error(y_test, pred_lasso))

melhor r2 score: 0.134062064100828
melhor alpha: 0.01
RMSE com o alpha escolhido: 2.20860448083319


##6.8. Inclusão das provas Ledor - 100 dimensões

###6.8.1. Regressão linear

In [ ]:
enem_data_ledor_100 = pd.read_pickle("/content/drive/MyDrive/MED-enem/cleaned_data_ledor_vectors_100.pkl").dropna()
enem_data_ledor_100.head()

,numero_questao,enunciado,alternativas,nu_param_B,gabarito,year,enunciado_limpo,alternativas_limpo,vetor_enunciado,vetor_media
0,1,A atmosfera terrestre é composta pelos gases n...,A: reduzir o calor irradiado pela Terra median...,-1.70677,C,2009,absorver acontece alternativa anos aquecendo a...,A: calor industrialização irradiado mediante p...,"[[-0.057471, 0.039134, -0.200435, -0.474476, -...","[0.04117549896267626, -0.06806348544983741, 0...."
1,2,Analise a figura. Supondo que seja necessário ...,A: Concentração média de álcool no sangue ao l...,0.62043,D,2009,alternativa analise dar figura melhor necessár...,A: concentração dia longo média sangue álcool;...,"[[0.470062, -0.2266, 0.16833, 0.139253, 0.1976...","[0.15159045354547826, -0.08034172671085055, 0...."
2,3,Estima-se que haja atualmente no mundo 40 milh...,"A: induzir a imunidade, para proteger o organi...",2.07704,A,2009,acompanhamento aids antiviral atualmente caren...,A: contaminação imunidade induzir organismo pr...,"[[0.046831, -0.536603, 0.054299, 0.036113, 0.1...","[0.06621599993343617, 0.027898976591491507, 0...."
3,4,"Em um experimento, preparou-se um conjunto de ...",A: os genótipos e os fenótipos idênticos.; B: ...,0.11500,B,2009,alguns amareladas apresentaram apresentava apó...,A: fenótipos genótipos idênticos; B: diferente...,"[[-0.304121, -0.076877, -0.194382, 0.040576, -...","[-0.008363717363980144, -0.04735812924515743, ..."
4,5,"Na linha de uma tradição antiga, o astrônomo g...",A: Ptolomeu apresentou as ideias mais valiosas...,0.21694,E,2009,afirmar afirmou alemão anos antiga astronômico...,A: antigas apresentou ideias ptolomeu serem tr...,"[[0.234921, 0.082011, -0.108453, -0.132333, 0....","[0.03087573888225724, -0.059190621633298586, 0..."


In [ ]:
# Coletando os dados
X = [np.array(embedding) for embedding in enem_data_ledor_100["enunciado_embbedings_word2vec"]]
y = enem_data_ledor_100["dificuldade"]

In [ ]:
# Aplicando Transformações
add_list = [(y.min() * (-1)) + 1] * len(y)
y = y + add_list

# Aplicando Boxcox
y, best_lambda = stats.boxcox(y)
print(best_lambda)

0.629376905725391


In [ ]:
# Separando em conjunto de treino e teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

In [ ]:
# Adicionando constante
X_train = sm.add_constant(X_train)
X_test = sm.add_constant(X_test)

In [ ]:
# Criando modelo e realizando predição
model = sm.OLS(y_train, X_train).fit()
pred = model.predict(X_test)

In [ ]:
# Cálculo do RMSE
rms = root_mean_squared_error(y_test, pred)
print('RMSE', rms)

RMSE 0.56786166889346


In [ ]:
print(model.summary(alpha = 0.05))

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.296
Model:                            OLS   Adj. R-squared:                  0.094
Method:                 Least Squares   F-statistic:                     1.468
Date:                Fri, 23 May 2025   Prob (F-statistic):            0.00614
Time:                        21:18:08   Log-Likelihood:                -273.25
No. Observations:                 451   AIC:                             748.5
Df Residuals:                     350   BIC:                             1164.
Df Model:                         100                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          1.8237      0.530      3.441      0.0

###6.8.2. Regressão Lasso

In [ ]:
# Coletando os dados
X = [np.array(embedding) for embedding in enem_data_ledor_100["enunciado_embbedings_word2vec"]]
y = enem_data_ledor_100["dificuldade"]

In [ ]:
# Aplicando Transformações
add_list = [(y.min() * (-1)) + 1] * len(y)
y = y + add_list

# Aplicando Boxcox
y, best_lambda = stats.boxcox(y)
print(best_lambda)

0.629376905725391


In [ ]:
# Separando em conjunto de treino e teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

In [ ]:
model_lasso = LassoCV(alphas=[0.0001, 0.001, 0.01, 0.1, 1, 10], random_state=0).fit(
    X_train, y_train
)
pred_lasso = model_lasso.predict(X_test)

print('melhor r2 score:', model_lasso.score(X_train, y_train))
print('melhor alpha:', model_lasso.alpha_)
print('RMSE com o alpha escolhido:', root_mean_squared_error(y_test, pred_lasso))

melhor r2 score: 0.16727413807662184
melhor alpha: 0.001
RMSE com o alpha escolhido: 0.5050435643572783


##6.9. Inclusão das provas Ledor - 50 dimensões

###6.9.1 Regressão linear

In [ ]:
enem_data_ledor_50 = pd.read_pickle("/content/drive/MyDrive/MED-enem/cleaned_data_ledor_vectors_50.pkl").dropna()
enem_data_ledor_50.head()

,numero_questao,enunciado,alternativas,nu_param_B,gabarito,year,enunciado_limpo,alternativas_limpo,vetor_enunciado,vetor_media
0,1,A atmosfera terrestre é composta pelos gases n...,A: reduzir o calor irradiado pela Terra median...,-1.70677,C,2009,absorver acontece alternativa anos aquecendo a...,A: calor industrialização irradiado mediante p...,"[[0.338463, -0.080108, 0.262583, -0.003842, -0...","[0.10937313253388685, 0.1518117644762456, 0.08..."
1,2,Analise a figura. Supondo que seja necessário ...,A: Concentração média de álcool no sangue ao l...,0.62043,D,2009,alternativa analise dar figura melhor necessár...,A: concentração dia longo média sangue álcool;...,"[[0.115119, -0.029904, 0.020403, 0.350273, 0.1...","[0.001039726659655571, -0.03333791112527251, 0..."
2,3,Estima-se que haja atualmente no mundo 40 milh...,"A: induzir a imunidade, para proteger o organi...",2.07704,A,2009,acompanhamento aids antiviral atualmente caren...,A: contaminação imunidade induzir organismo pr...,"[[0.072015, -0.296274, 0.120401, 0.09635, 0.04...","[0.02792688441831012, 0.10160078826295428, 0.0..."
3,4,"Em um experimento, preparou-se um conjunto de ...",A: os genótipos e os fenótipos idênticos.; B: ...,0.11500,B,2009,alguns amareladas apresentaram apresentava apó...,A: fenótipos genótipos idênticos; B: diferente...,"[[0.108379, -0.268218, -0.0819, -0.138854, -0....","[-0.033945256138208486, 0.09432189668027255, 0..."
4,5,"Na linha de uma tradição antiga, o astrônomo g...",A: Ptolomeu apresentou as ideias mais valiosas...,0.21694,E,2009,afirmar afirmou alemão anos antiga astronômico...,A: antigas apresentou ideias ptolomeu serem tr...,"[[0.13255, 0.158937, -0.179955, -0.278152, -0....","[0.05192682555660713, 0.05507281089903436, 0.0..."


In [ ]:
# Coletando os dados
X = [np.array(embedding) for embedding in enem_data_ledor_50["enunciado_embbedings_word2vec"]]
y = enem_data_ledor_50["dificuldade"]

In [ ]:
# Aplicando Transformações
add_list = [(y.min() * (-1)) + 1] * len(y)
y = y + add_list

# Aplicando Boxcox
y, best_lambda = stats.boxcox(y)
print(best_lambda)

0.629376905725391


In [ ]:
# Separando em conjunto de treino e teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

In [ ]:
# Adicionando constante
X_train = sm.add_constant(X_train)
X_test = sm.add_constant(X_test)

In [ ]:
# Criando modelo e realizando predição
model = sm.OLS(y_train, X_train).fit()
pred = model.predict(X_test)

In [ ]:
# Cálculo do RMSE
rms = root_mean_squared_error(y_test, pred)
print('RMSE', rms)

RMSE 0.5207235087061666


In [ ]:
print(model.summary(alpha = 0.05))

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.167
Model:                            OLS   Adj. R-squared:                  0.063
Method:                 Least Squares   F-statistic:                     1.608
Date:                Fri, 23 May 2025   Prob (F-statistic):            0.00755
Time:                        21:18:12   Log-Likelihood:                -310.95
No. Observations:                 451   AIC:                             723.9
Df Residuals:                     400   BIC:                             933.6
Df Model:                          50                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          1.8708      0.408      4.587      0.0

###6.9.2. Regressão Lasso

In [ ]:
# Coletando os dados
X = [np.array(embedding) for embedding in enem_data_ledor_50["enunciado_embbedings_word2vec"]]
y = enem_data_ledor_50["dificuldade"]

In [ ]:
# Aplicando Transformações
add_list = [(y.min() * (-1)) + 1] * len(y)
y = y + add_list

# Aplicando Boxcox
y, best_lambda = stats.boxcox(y)
print(best_lambda)

0.629376905725391


In [ ]:
# Separando em conjunto de treino e teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

In [ ]:
model_lasso = LassoCV(alphas=[0.0001, 0.001, 0.01, 0.1, 1, 10], random_state=0).fit(
    X_train, y_train
)
pred_lasso = model_lasso.predict(X_test)

print('melhor r2 score:', model_lasso.score(X_train, y_train))
print('melhor alpha:', model_lasso.alpha_)
print('RMSE com o alpha escolhido:', root_mean_squared_error(y_test, pred_lasso))

melhor r2 score: 0.12256955400055136
melhor alpha: 0.001
RMSE com o alpha escolhido: 0.5051223890309581
